# 리뷰 텍스트 전처리 - Step 2: 중복 처리

`reviews_step1_cleaned.csv`를 입력으로 받아 같은 작성자의 다중 리뷰 작성을 식별하고 처리한다.

## 처리 전략

| Step | 조건 | 처리 |
|------|------|------|
| 0 | 작성자 = `'-'` (탈퇴/닉변) | 각 행을 고유 ID로 분리 (서로 다른 사람으로 처리) |
| 1 | 리뷰타입 = `month` | 다중 작성 검토에서 제외 (다른 시점의 정보) |
| 2-A | 같은 작성자 + 같은 상품 + **같은 옵션** + 같은 타입 + 24h 이내 | 한글_글자수 많은 1개 보존 |
| 2-B | 같은 작성자 + 같은 상품 + **다른 옵션** + 같은 타입 + 24h 이내 + Jaccard ≥ 0.85 | 1개 보존, `multi_option_dup` 플래그 |
| 2-C | 같은 작성자 + 같은 상품 + **다른 옵션** + 같은 타입 + 24h 이내 + Jaccard < 0.85 | 둘 다 보존, `multi_option_unique` 플래그 |
| 3 | 다른 타입 + 24h 이내 + Jaccard ≥ 0.85 | 1개 보존, `multi_type_dup` 플래그 |
| 4 | 다른 타입 + 24h 이내 + Jaccard < 0.85 | 둘 다 보존, `multi_type_unique` 플래그 |
| 5 | 24h 초과 | 재구매 가능성 분류, `possible_repurchase` 플래그 |

**Jaccard 유사도**: 문자 bi-gram 기반. 텍스트 길이 15자 미만이면 우연 일치 방지를 위해 완전일치만 인정.

**24h 세션 분리**: 같은 (작성자, 상품) 안에서 작성일 정렬 후 인접 리뷰 간 시간차 ≤ 24h면 동일 세션으로 묶음.


---
## 0. 데이터 로드

In [14]:
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

INPUT_PATH = "reviews_step1_cleaned.csv"
TEXT_COL = "리뷰내용_norm"   # 이미 정규화된 텍스트 컬럼 (공백·문장부호 제거됨)

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"로드 완료: {len(df):,}건")
print(f"컬럼 수: {df.shape[1]}개")

로드 완료: 685,042건
컬럼 수: 57개


In [15]:
# 작성일 datetime 변환
df['작성일'] = pd.to_datetime(df['작성일'], errors='coerce')

# 작성일 결측 확인
n_nat = df['작성일'].isna().sum()
print(f"작성일 결측: {n_nat:,}건")

# 텍스트 결측 → 빈 문자열로
df[TEXT_COL] = df[TEXT_COL].fillna('').astype(str)

작성일 결측: 0건


---
## STEP 0. 탈퇴/닉변 작성자 처리

작성자가 `'-'`인 경우, 무신사 시스템상 익명 처리된 케이스이므로 각 행을 **서로 다른 사람**으로 취급한다.
→ 이후 중복 검토에서 자동으로 제외됨.

In [16]:
# '-' 작성자에 고유 ID 부여
df['작성자_norm'] = df['작성자'].astype(str)
mask_anon = df['작성자_norm'].str.strip() == "'-"

# 익명 행은 인덱스 기반 고유 ID로 변환
df.loc[mask_anon, '작성자_norm'] = (
    '_anon_' + df.loc[mask_anon].index.astype(str)
)

print(f"탈퇴/닉변 작성자: {mask_anon.sum():,}건 → 고유 ID로 변환")
print(f"고유 작성자 수: {df['작성자_norm'].nunique():,}명")

탈퇴/닉변 작성자: 3,975건 → 고유 ID로 변환
고유 작성자 수: 349,096명


---
## STEP 1. month 타입 분리

리뷰타입이 `month`인 리뷰는 한 달 사용 후 작성하는 후기로, 다른 타입(일반/스타일)과 시점·결이 다르다.
→ 중복 검토 대상에서 제외하고 그대로 보존.

In [17]:
print("리뷰타입 분포:")
print(df['리뷰타입'].value_counts(dropna=False))

mask_month = df['리뷰타입'] == 'month'
df_month = df[mask_month].copy()
df_main = df[~mask_month].copy()

print(f"\nmonth 리뷰: {len(df_month):,}건 (중복 검토 제외)")
print(f"main 리뷰: {len(df_main):,}건 (중복 검토 대상)")

리뷰타입 분포:
리뷰타입
goods         376942
photo         109292
style         104397
general        68881
month          25088
experience       442
Name: count, dtype: int64

month 리뷰: 25,088건 (중복 검토 제외)
main 리뷰: 659,954건 (중복 검토 대상)


---
## 보조 컬럼

- **옵션키**: `(구매사이즈, 구매상세)` 튜플. NaN은 `None`으로 통일하여 옵션 정보 없는 행끼리 동일 옵션으로 인정.
- **보존 기준**: Step 1에서 만들어진 `한글_글자수` 컬럼을 그대로 활용. 그룹 내에서 한글 글자가 많은 리뷰가 base.

In [18]:
# 옵션키 생성: (사이즈, 상세) 튜플, NaN은 None
def make_option_key(s, d):
    s_v = s if pd.notna(s) else None
    d_v = d if pd.notna(d) else None
    return (s_v, d_v)

df_main['옵션키'] = [
    make_option_key(s, d)
    for s, d in zip(df_main['구매사이즈'], df_main['구매상세'])
]

print(f"옵션키 고유값: {df_main['옵션키'].nunique():,}개")
print(f"한글_글자수 통계:\n{df_main['한글_글자수'].describe()}")

옵션키 고유값: 1,570개
한글_글자수 통계:
count    659954.000000
mean         33.483541
std          21.750184
min           0.000000
25%          22.000000
50%          27.000000
75%          36.000000
max        1051.000000
Name: 한글_글자수, dtype: float64


---
# 하린 수정 부분
## COSINE_JACCARD_THRESHOLD 선정

`process_group`에서 두 리뷰를 **중복**으로 볼지 **독립 리뷰**로 볼지 가르는 기준값을 데이터 기반으로 결정한다.

### 분석 방법
- 동일 (작성자, 상품, 옵션, 리뷰타입) 그룹 내 **24h 이내 페어** 전수 추출
- TF-IDF `char_wb` 2~3-gram 기반 코사인 유사도 계산
- 구간별 분포 시각화 + 실제 텍스트 샘플 육안 확인

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# ── 임계값  ────────────────────────────────────────────────
'''
JACCARD_THRESHOLD       = 0.7
COSINE_THRESHOLD        = 0.8
SHORT_JACCARD_THRESHOLD = 0.6   # 짧은 텍스트 (조정 중)
SHORT_COSINE_THRESHOLD  = 0.65
SHORT_TEXT_LIMIT        = 15
'''

# ── 임계값3  ────────────────────────────────────────────────
JACCARD_THRESHOLD       = 0.45
COSINE_THRESHOLD        = 0.5
SHORT_JACCARD_THRESHOLD = 0.6   # 짧은 텍스트 (조정 중)
SHORT_COSINE_THRESHOLD  = 0.65
SHORT_TEXT_LIMIT        = 15

# ── Jaccard 유사도 ─────────────────────────────────────────
def char_bigrams(s):
    return {s[i:i+2] for i in range(len(s) - 1)} if len(s) >= 2 else {s}

def jaccard_sim(s1, s2):
    g1, g2 = char_bigrams(str(s1)), char_bigrams(str(s2))
    union = g1 | g2
    return round(len(g1 & g2) / len(union), 4) if union else 0.0

# ── 하이브리드 판정 ────────────────────────────────────────
def is_dup_hybrid(s1, s2, jac, cos):
    if len(s1) < SHORT_TEXT_LIMIT or len(s2) < SHORT_TEXT_LIMIT:
        return (jac >= SHORT_JACCARD_THRESHOLD) or (cos >= SHORT_COSINE_THRESHOLD)
    return (jac >= JACCARD_THRESHOLD) or (cos >= COSINE_THRESHOLD)

# ── 24h 이내 페어 추출 ─────────────────────────────────────
quasi_key = ['goodsNo', '작성자_norm', '구매사이즈', '구매상세', '리뷰타입']

candidates = []
for _, grp in df_main.groupby(quasi_key):
    if len(grp) < 2:
        continue
    grp = grp.sort_values('작성일')
    idxs  = list(grp.index)
    dates = grp['작성일'].tolist()
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            diff_h = abs((dates[j] - dates[i]).total_seconds()) / 3600
            if diff_h <= 24:
                candidates.append({
                    '텍스트A'   : str(grp.at[idxs[i], TEXT_COL] or ''),
                    '텍스트B'   : str(grp.at[idxs[j], TEXT_COL] or ''),
                    '시간차(h)' : round(diff_h, 2),
                })

cand_df = pd.DataFrame(candidates)
print(f"24h 이내 페어 수: {len(cand_df):,}")

# ── Cosine 유사도 (batch) ──────────────────────────────────
vec_diag = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3), min_df=1)
all_texts = pd.concat([cand_df['텍스트A'], cand_df['텍스트B']]).unique()
vec_diag.fit(all_texts)

mat_a = normalize(vec_diag.transform(cand_df['텍스트A'].tolist()))
mat_b = normalize(vec_diag.transform(cand_df['텍스트B'].tolist()))
cand_df['코사인'] = np.round(
    np.array(mat_a.multiply(mat_b).sum(axis=1)).flatten(), 4
)

# ── Jaccard 유사도 (batch) ─────────────────────────────────
cand_df['자카드'] = [
    jaccard_sim(a, b)
    for a, b in zip(cand_df['텍스트A'], cand_df['텍스트B'])
]

# ── 하이브리드 판정 ────────────────────────────────────────
cand_df['중복'] = [
    is_dup_hybrid(a, b, jac, cos)
    for a, b, jac, cos in zip(
        cand_df['텍스트A'], cand_df['텍스트B'],
        cand_df['자카드'], cand_df['코사인']
    )
]
cand_df['짧은텍스트'] = cand_df.apply(
    lambda r: len(r['텍스트A']) < SHORT_TEXT_LIMIT or len(r['텍스트B']) < SHORT_TEXT_LIMIT,
    axis=1
)

# ── 전체 통계 ──────────────────────────────────────────────
print("\n[Cosine 기술통계]")
print(cand_df['코사인'].describe())
print("\n[Jaccard 기술통계]")
print(cand_df['자카드'].describe())

total = len(cand_df)
dup_total   = cand_df['중복'].sum()
jac_only    = ((cand_df['자카드'] >= JACCARD_THRESHOLD) & (cand_df['코사인'] < COSINE_THRESHOLD)).sum()
cos_only    = ((cand_df['자카드'] < JACCARD_THRESHOLD)  & (cand_df['코사인'] >= COSINE_THRESHOLD)).sum()
both        = ((cand_df['자카드'] >= JACCARD_THRESHOLD) & (cand_df['코사인'] >= COSINE_THRESHOLD)).sum()

print(f"\n[하이브리드 중복 판정]")
print(f"  전체 페어:             {total:,}")
print(f"  중복 판정:             {dup_total:,}  ({dup_total/total*100:.1f}%)")
print(f"  ├ Jaccard 단독으로만:  {jac_only:,}")
print(f"  ├ Cosine  단독으로만:  {cos_only:,}")
print(f"  └ 둘 다 해당:          {both:,}")
print(f"  짧은 텍스트 페어:      {cand_df['짧은텍스트'].sum():,}")

# ── 코사인 구간별 분포 + 샘플 ─────────────────────────────
bins = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.01]
print("\n[Cosine 구간별 분포]")
print(pd.cut(cand_df['코사인'], bins=bins, include_lowest=True)
        .value_counts().sort_index())

print("\n[Jaccard 구간별 분포]")
print(pd.cut(cand_df['자카드'], bins=bins, include_lowest=True)
        .value_counts().sort_index())

# ── 구간별 샘플 (짧은 텍스트 / 긴 텍스트 분리) ─────────────
ranges = [(0.0,0.1),(0.1,0.2),(0.2,0.3),(0.3,0.4),(0.4,0.5),
          (0.5,0.6),(0.6,0.7),(0.7,0.8),(0.8,0.9),(0.9,1.01)]

for label, mask in [("짧은 텍스트 (< 15자)", cand_df['짧은텍스트']),
                    ("긴 텍스트  (≥ 15자)", ~cand_df['짧은텍스트'])]:
    sub = cand_df[mask]
    print(f"\n\n{'#'*70}")
    print(f"# {label}  (전체 {len(sub):,}쌍)")
    print(f"{'#'*70}")

    for lo, hi in ranges:
        sample = sub[(sub['코사인'] >= lo) & (sub['코사인'] < hi)]
        if len(sample) == 0:
            continue
        print(f"\n{'='*70}")
        print(f"  Cosine {lo:.1f}~{hi:.2f}  ({len(sample):,}쌍)")
        print('='*70)
        for _, row in sample.head(5).iterrows():
            dup_mark = "✅ 중복" if row['중복'] else "❌ 독립"
            print(f"  Cos={row['코사인']}  Jac={row['자카드']}  시간차={row['시간차(h)']}h  {dup_mark}")
            print(f"  A: {row['텍스트A'][:80]}")
            print(f"  B: {row['텍스트B'][:80]}")
            print()

24h 이내 페어 수: 626

[Cosine 기술통계]
count    626.000000
mean       0.504241
std        0.442542
min        0.000000
25%        0.060125
50%        0.329550
75%        1.000000
max        1.000000
Name: 코사인, dtype: float64

[Jaccard 기술통계]
count    626.000000
mean       0.500781
std        0.437657
min        0.000000
25%        0.075000
50%        0.295150
75%        1.000000
max        1.000000
Name: 자카드, dtype: float64

[하이브리드 중복 판정]
  전체 페어:             626
  중복 판정:             294  (47.0%)
  ├ Jaccard 단독으로만:  1
  ├ Cosine  단독으로만:  5
  └ 둘 다 해당:          288
  짧은 텍스트 페어:      1

[Cosine 구간별 분포]
코사인
(-0.001, 0.1]    216
(0.1, 0.2]        58
(0.2, 0.3]        33
(0.3, 0.4]        18
(0.4, 0.5]         8
(0.5, 0.6]         9
(0.6, 0.7]        10
(0.7, 0.8]         9
(0.8, 0.9]         7
(0.9, 1.01]      258
Name: count, dtype: int64

[Jaccard 구간별 분포]
자카드
(-0.001, 0.1]    209
(0.1, 0.2]        80
(0.2, 0.3]        25
(0.3, 0.4]        18
(0.4, 0.5]         8
(0.5, 0.6]        11
(0.6, 0.7]  

### 결과 및 결론 (v2 하이브리드)

| 구간 | Jaccard | Cosine | 판정 |
|------|---------|--------|------|
| 확실히 다름 | < 0.7 | < 0.8 | 독립 리뷰 (정상) |
| 내용 유사 (일반) | ≥ 0.7 OR | ≥ 0.8 | 중복/준중복 |
| 짧은 텍스트 (< 15자) | ≥ 0.9 OR | ≥ 0.95 | 엄격 기준 적용 |

- 이봉 분포 확인 → 0~0.1 구간과 0.9~1.0 구간에 페어 집중
- 0.5 미만 샘플도 비슷한 케이스 존재 → 단일 임계값의 한계 확인
- **리뷰내용_norm** 사용으로 정규화 중복 처리 불필요 (이미 공백·문장부호 제거됨)
- Jaccard와 Cosine의 OR 조건으로 각 지표의 약점 보완
  - Jaccard: 단어 집합 일치 강점 (복붙 패턴에 강함)
  - Cosine: 어미 변화·오타 흡수 강점 (표현 변형에 강함)
- 짧은 텍스트 (정규화 후 15자 미만): 한두 글자 차이로 의미가 반대가 될 수 있어 **더 높은 임계값** 적용
  - ex) "좋아요" vs "안좋아요" → 의미 반대 → 독립 처리

```
JACCARD_THRESHOLD       = 0.7    # 일반 텍스트
COSINE_THRESHOLD        = 0.8    # 일반 텍스트
SHORT_JACCARD_THRESHOLD = 0.9    # 짧은 텍스트 (엄격)
SHORT_COSINE_THRESHOLD  = 0.95   # 짧은 텍스트 (엄격)
SHORT_TEXT_LIMIT        = 15     # 정규화 후 글자 수 기준
```

In [12]:
"""
하이브리드 유사도 엔진 (v2)
─────────────────────────────────────────────────────
변경 사항 (v1 대비):
  1. 정규화 추가  : 유사도 계산 전 문장부호·공백 제거
                    (리뷰내용_norm은 이미 정규화됨 → 별도 처리 불필요)
  2. Jaccard 추가 : character bi-gram 기반
  3. 하이브리드   : (Jaccard >= 0.7) OR (Cosine >= 0.8) → 중복
  4. 짧은 텍스트  : 정규화 후 15자 미만이면 임계값을 높여 엄격하게 판단
                    (Jaccard >= 0.9) OR (Cosine >= 0.95)
                    이유: 짧은 텍스트는 한두 글자만 달라도 의미가 완전히 바뀔 수 있어
                    더 높은 유사도일 때만 중복으로 판정해야 오분류를 줄일 수 있음
                    ex) "좋아요" vs "안좋아요" → 의미 반대 → 독립으로 처리
  5. 벡터라이저   : char_wb 2~3-gram (정규화된 텍스트에 적용)

임계값 근거:
  - Jaccard 0.7 / Cosine 0.8  : 일반 텍스트 기준
  - Jaccard 0.9 / Cosine 0.95 : 짧은 텍스트 기준 (더 엄격)
─────────────────────────────────────────────────────
"""

import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── 임계값 ──────────────────────────────────────────────────
JACCARD_THRESHOLD       = 0.7    # 일반 텍스트 Jaccard 기준
COSINE_THRESHOLD        = 0.8    # 일반 텍스트 Cosine  기준
SHORT_JACCARD_THRESHOLD = 0.9    # 짧은 텍스트 Jaccard 기준 (엄격)
SHORT_COSINE_THRESHOLD  = 0.95   # 짧은 텍스트 Cosine  기준 (엄격)
SHORT_TEXT_LIMIT        = 15     # 정규화 후 글자 수 기준 (미만이면 짧은 텍스트)

# ── 벡터라이저 (char_wb 2~3-gram, 정규화된 텍스트에 적용) ───
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3), min_df=1)

# ── 1. 정규화 ──────────────────────────────────────────────
def normalize_for_sim(text: str) -> str:
    """유사도 계산 전용 정규화: 문장부호·공백 제거.
    리뷰내용_norm은 이미 정규화돼 있으므로 별도 호출 불필요.
    raw 텍스트(리뷰내용_clean 등) 비교 시에만 사용.
    """
    if not text:
        return ""
    text = re.sub(r'[\s\.\,\!\?\~\^\;\:\-\_\(\)\[\]\"\']+', '', str(text))
    return text.strip()


# ── 2. Jaccard 유사도 (character bi-gram) ──────────────────
def char_bigrams(s: str) -> set:
    return {s[i:i+2] for i in range(len(s) - 1)} if len(s) >= 2 else {s}

def jaccard_sim(s1: str, s2: str) -> float:
    """character bi-gram Jaccard. 리뷰내용_norm은 이미 정규화돼 있음."""
    n1 = str(s1 or '')
    n2 = str(s2 or '')
    if not n1 or not n2:
        return 0.0
    g1, g2 = char_bigrams(n1), char_bigrams(n2)
    union = g1 | g2
    return round(len(g1 & g2) / len(union), 4) if union else 0.0


# ── 3. Cosine 유사도 (TF-IDF char_wb) ───────────────────────
def cosine_sim(s1: str, s2: str) -> float:
    """TF-IDF char_wb 코사인 유사도. 리뷰내용_norm은 이미 정규화돼 있음."""
    n1 = str(s1 or '')
    n2 = str(s2 or '')
    if not n1 or not n2:
        return 0.0
    try:
        mat = vec.fit_transform([n1, n2])
        return round(float(cosine_similarity(mat[0:1], mat[1:])[0][0]), 4)
    except Exception:
        return 0.0


# ── 4. 하이브리드 중복 판단 ─────────────────────────────────
def is_duplicate(s1: str, s2: str):
    """
    (중복여부, jaccard, cosine) 반환.

    짧은 텍스트 (정규화 후 15자 미만):
      → 임계값 높여서 엄격하게 판단
      → (Jaccard >= 0.9) OR (Cosine >= 0.95)
      → 이유: 짧은 텍스트는 한두 글자 차이로 의미가 반대가 될 수 있음
              ex) "좋아요" vs "안좋아요"

    일반 텍스트:
      → (Jaccard >= 0.7) OR (Cosine >= 0.8)
    """
    n1 = str(s1 or '')
    n2 = str(s2 or '')

    jac = jaccard_sim(n1, n2)
    cos = cosine_sim(n1, n2)

    # 짧은 텍스트: 더 높은 임계값 적용
    if len(n1) < SHORT_TEXT_LIMIT or len(n2) < SHORT_TEXT_LIMIT:
        dup = (jac >= SHORT_JACCARD_THRESHOLD) or (cos >= SHORT_COSINE_THRESHOLD)
    else:
        dup = (jac >= JACCARD_THRESHOLD) or (cos >= COSINE_THRESHOLD)

    return dup, jac, cos


# ── 5. 동작 확인 ─────────────────────────────────────────────
print("=" * 70)
print("[하이브리드 유사도 테스트]")
print(f"  일반:  Jaccard >= {JACCARD_THRESHOLD} OR Cosine >= {COSINE_THRESHOLD}")
print(f"  짧은:  Jaccard >= {SHORT_JACCARD_THRESHOLD} OR Cosine >= {SHORT_COSINE_THRESHOLD}  (정규화 후 {SHORT_TEXT_LIMIT}자 미만)")
print("=" * 70)
print(f"  {'케이스':<32} {'Jac':>6} {'Cos':>6} {'짧은?':>5} {'중복?'}")
print(f"  {'-'*65}")

test_cases = [
    ("문장부호만 다름",        '정말좋아요잘입을게요',              '정말좋아요잘입을게요'),
    ("공백만 다름",            '가성비좋고핏도정사이즈',            '가성비좋고핏도정사이즈'),
    ("부분 유사",              '두께감적당하고핏좋습니다',           '두께감적당하고색감예쁘네요'),
    ("완전 다름",              '정말좋아요잘입을게요',              '사이즈가너무작아서환불'),
    ("짧은 동일",              '좋아요',                            '좋아요'),
    ("짧은 의미 반대",         '좋아요',                            '안좋아요'),
    ("짧은 유사 (잡아야 함)",  '따뜻해요',                         '따뜻합니다'),
    ("페어3 케이스",           '제품나쁘지않네요크기적당하고따뜻한거같아요',
                                '제품잘받았습니다크기적당하고따뜻하네요굿'),
]
for name, a, b in test_cases:
    dup, jac, cos = is_duplicate(a, b)
    is_short = len(a) < SHORT_TEXT_LIMIT or len(b) < SHORT_TEXT_LIMIT
    short_mark = "✓" if is_short else ""
    mark = "✅ 중복" if dup else "❌ 독립"
    print(f"  {name:<32} {jac:>6.3f} {cos:>6.3f} {short_mark:>5}   {mark}")

[하이브리드 유사도 테스트]
  일반:  Jaccard >= 0.5 OR Cosine >= 0.6
  짧은:  Jaccard >= 0.7 OR Cosine >= 0.7  (정규화 후 15자 미만)
  케이스                                 Jac    Cos   짧은? 중복?
  -----------------------------------------------------------------
  문장부호만 다름                          1.000  1.000     ✓   ✅ 중복
  공백만 다름                            1.000  1.000     ✓   ✅ 중복
  부분 유사                             0.353  0.410     ✓   ❌ 독립
  완전 다름                             0.000  0.072     ✓   ❌ 독립
  짧은 동일                             1.000  1.000     ✓   ✅ 중복
  짧은 의미 반대                          0.667  0.642     ✓   ❌ 독립
  짧은 유사 (잡아야 함)                     0.167  0.326     ✓   ❌ 독립
  페어3 케이스                           0.300  0.349         ❌ 독립


In [13]:
# ── 검증 1: 정규화 전후 유사도 비교 ────────────────────────────
print("=" * 65)
print("[검증 1] 정규화 전후 유사도 비교")
print("=" * 65)

def cosine_sim_raw(s1, s2):
    """정규화 없이 원본 텍스트로 코사인 계산 (비교용)"""
    if not s1 or not s2:
        return 0.0
    try:
        mat = vec.fit_transform([str(s1), str(s2)])
        return round(float(cosine_similarity(mat[0:1], mat[1:])[0][0]), 4)
    except:
        return 0.0

compare_cases = [
    ("문장부호만 다름", '정말 좋아요. 잘 입을게요!',  '정말 좋아요 잘 입을게요'),
    ("공백만 다름",     '가성비 좋고 핏도 정사이즈',   '가성비좋고핏도정사이즈'),
    ("페어3 케이스",    '제품 나쁘지 않네요 크기 적당하고 따뜻한거같아요',
                        '제품 잘 받았습니다 크기 적당하고 따뜻하네요 굿'),
]
print(f"  {'케이스':<20} {'정규화 전 Cos':>14} {'정규화 후 Cos':>14} {'변화':>8}")
print(f"  {'-'*60}")
for name, a, b in compare_cases:
    before = cosine_sim_raw(a, b)
    after  = cosine_sim(a, b)
    diff   = after - before
    print(f"  {name:<20} {before:>14.3f} {after:>14.3f} {diff:>+8.3f}")


# ── 검증 2: 짧은 텍스트 엄격 임계값 확인 ────────────────────
print()
print("=" * 65)
print("[검증 2] 짧은 텍스트 임계값 비교")
print(f"  일반:  Jaccard >= {JACCARD_THRESHOLD} OR Cosine >= {COSINE_THRESHOLD}")
print(f"  짧은:  Jaccard >= {SHORT_JACCARD_THRESHOLD} OR Cosine >= {SHORT_COSINE_THRESHOLD}")
print("=" * 65)
short_cases = [
    ("짧은 동일",              '좋아요',    '좋아요'),
    ("짧은 의미 반대",         '좋아요',    '안좋아요'),
    ("짧은 유사 (잡아야 함)",  '따뜻해요',  '따뜻합니다'),
    ("짧은 어미만 다름",       '크기 딱 맞아요', '크기 딱 맞습니다'),
]
print(f"  {'케이스':<26} {'Jac':>6} {'Cos':>6}   {'일반기준':>8} {'짧은기준':>8}")
print(f"  {'-'*65}")
for name, a, b in short_cases:
    dup, jac, cos = is_duplicate(a, b)
    general_dup = (jac >= JACCARD_THRESHOLD) or (cos >= COSINE_THRESHOLD)
    short_dup   = (jac >= SHORT_JACCARD_THRESHOLD) or (cos >= SHORT_COSINE_THRESHOLD)
    print(f"  {name:<26} {jac:>6.3f} {cos:>6.3f}   {'✅' if general_dup else '❌':>8} {'✅' if short_dup else '❌':>8}")


# ── 검증 3: 하이브리드 경계 샘플 (24h 이내 페어) ────────────
print()
print("=" * 65)
print("[검증 3] 하이브리드 경계 샘플 (24h 이내 페어 기준)")
print("=" * 65)

if 'df_main' in dir():
    quasi_key = ['goodsNo', '작성자_norm', '구매사이즈', '구매상세', '리뷰타입']
    pairs = []
    for _, grp in df_main.groupby(quasi_key):
        if len(grp) < 2:
            continue
        grp = grp.sort_values('작성일')
        idxs = list(grp.index)
        dates = grp['작성일'].tolist()
        for i in range(len(idxs)):
            for j in range(i+1, len(idxs)):
                diff_h = abs((dates[j]-dates[i]).total_seconds())/3600
                if diff_h <= 24:
                    t1 = str(grp.at[idxs[i], TEXT_COL] or '')
                    t2 = str(grp.at[idxs[j], TEXT_COL] or '')
                    dup, jac, cos = is_duplicate(t1, t2)
                    pairs.append({'텍스트A': t1, '텍스트B': t2,
                                   '시간차(h)': round(diff_h,2),
                                   'Jaccard': jac, 'Cosine': cos, '중복': dup})
    pairs_df = pd.DataFrame(pairs)
    print(f"  전체 24h 이내 페어: {len(pairs_df):,}쌍")
    print(f"  중복 판정: {pairs_df['중복'].sum():,}쌍 ({pairs_df['중복'].mean()*100:.1f}%)")
    print(f"  독립 판정: {(~pairs_df['중복']).sum():,}쌍")

    # Jaccard로만 잡힌 케이스
    jac_only = pairs_df[(pairs_df['Jaccard']>=JACCARD_THRESHOLD) & (pairs_df['Cosine']<COSINE_THRESHOLD)]
    cos_only = pairs_df[(pairs_df['Jaccard']<JACCARD_THRESHOLD) & (pairs_df['Cosine']>=COSINE_THRESHOLD)]
    both     = pairs_df[(pairs_df['Jaccard']>=JACCARD_THRESHOLD) & (pairs_df['Cosine']>=COSINE_THRESHOLD)]
    print(f"\n  Jaccard 단독으로 잡힌 중복: {len(jac_only):,}쌍")
    print(f"  Cosine  단독으로 잡힌 중복: {len(cos_only):,}쌍")
    print(f"  둘 다 중복으로 판정:         {len(both):,}쌍")

    # Cosine이 단독으로 잡은 샘플 (Jaccard의 약점을 보완한 케이스)
    if len(cos_only) > 0:
        print(f"\n  [Cosine이 단독으로 잡은 샘플 - Jaccard 보완]")
        for _, row in cos_only.head(3).iterrows():
            print(f"    Jac={row['Jaccard']:.3f} Cos={row['Cosine']:.3f}")
            print(f"    A: {row['텍스트A'][:70]}")
            print(f"    B: {row['텍스트B'][:70]}")
            print()
else:
    print("  ⚠️  df_main이 없습니다. 앞 셀을 먼저 실행하세요.")


[검증 1] 정규화 전후 유사도 비교
  케이스                       정규화 전 Cos      정규화 후 Cos       변화
  ------------------------------------------------------------
  문장부호만 다름                      0.859          1.000   +0.141
  공백만 다름                        0.565          1.000   +0.435
  페어3 케이스                       0.774          0.349   -0.425

[검증 2] 짧은 텍스트 임계값 비교
  일반:  Jaccard >= 0.5 OR Cosine >= 0.6
  짧은:  Jaccard >= 0.7 OR Cosine >= 0.7
  케이스                           Jac    Cos       일반기준     짧은기준
  -----------------------------------------------------------------
  짧은 동일                       1.000  1.000          ✅        ✅
  짧은 의미 반대                    0.667  0.642          ✅        ❌
  짧은 유사 (잡아야 함)               0.167  0.326          ❌        ❌
  짧은 어미만 다름                   0.375  0.446          ❌        ❌

[검증 3] 하이브리드 경계 샘플 (24h 이내 페어 기준)
  전체 24h 이내 페어: 626쌍
  중복 판정: 286쌍 (45.7%)
  독립 판정: 340쌍

  Jaccard 단독으로 잡힌 중복: 3쌍
  Cosine  단독으로 잡힌 중복: 0쌍
  둘 다 중복으로 판정:         283쌍


---
## 24h 세션 분리

같은 (작성자, 상품) 안에서 작성일 순 정렬 → 인접 리뷰 간 시간차 ≤ 24h면 같은 세션 ID 부여.
서로 다른 세션은 **재구매 케이스(Step 5)** 로 분류된다.

In [9]:
# 정렬: 작성자 → 상품 → 작성일
df_main = df_main.sort_values(
    ['작성자_norm', 'goodsNo', '작성일']
).reset_index(drop=True)

# 그룹별 세션 ID 부여 (벡터화)
# 같은 (작성자, 상품) 안에서 인접 행 시간차가 24h 초과면 새 세션 시작
g = df_main.groupby(['작성자_norm', 'goodsNo'])
prev_time = g['작성일'].shift(1)
time_diff_h = (df_main['작성일'] - prev_time).dt.total_seconds() / 3600

# 새 세션 시작 시점 = 그룹 첫 행 OR 시간차 > 24h
new_session = (prev_time.isna()) | (time_diff_h > 24)
df_main['세션'] = new_session.groupby(
    [df_main['작성자_norm'], df_main['goodsNo']]
).cumsum().astype(int)

print(f"세션 부여 완료")
print(f"(작성자, 상품) 그룹 수: {g.ngroups:,}")
print(f"(작성자, 상품, 세션) 그룹 수: {df_main.groupby(['작성자_norm', 'goodsNo', '세션']).ngroups:,}")

세션 부여 완료
(작성자, 상품) 그룹 수: 469,991
(작성자, 상품, 세션) 그룹 수: 526,828


In [10]:
#pd.set_option('display.max_columns',None)
pd.options.display.max_columns = 30

In [11]:
# 같은 (작성자, 상품)에 세션이 2개 이상 → 재구매
session_per_pair = df_main.groupby(['작성자_norm', 'goodsNo'])['세션'].transform('nunique')
df_main['is_repurchase'] = session_per_pair > 1

n_repurchase_rows = df_main['is_repurchase'].sum()
n_repurchase_pairs = (session_per_pair > 1).groupby(
    [df_main['작성자_norm'], df_main['goodsNo']]
).first().sum()
print(f"재구매 행 수: {n_repurchase_rows:,}건")
print(f"재구매 (작성자, 상품) 쌍: {n_repurchase_pairs:,}쌍")

재구매 행 수: 129,495건
재구매 (작성자, 상품) 쌍: 50,181쌍


---
## 그룹 단위 중복 처리 (Step 2 ~ 4)

`(작성자_norm, goodsNo, 세션)` 단위로 묶고:

1. 그룹 크기 = 1 → 단독 리뷰 (그대로 keep)
2. 그룹 크기 ≥ 2:
   - 한글_글자수가 가장 많은 행을 **기준 리뷰**로 지정
   - 같은 그룹의 다른 리뷰 각각에 대해 옵션 일치 여부, 타입 일치 여부, cosine 유사도를 계산하여 케이스 분기

In [12]:
def process_group(group: pd.DataFrame, group_name: str, counter: list) -> pd.DataFrame:
    """
    같은 IDENTITY (goodsNo, 작성자_처리, 평점, 구매옵션, 리뷰타입) 그룹 내
    24h 이내 다중 리뷰 처리.

    분류 기준:
      - 리뷰내용 동일 (24h 이내)     → 중복
      - 유사도 높음  (24h 이내)      → 준중복 (대표 1개 보존)
      - 유사도 낮음  (24h 이내)      → 준중복_복수 (모두 유효)

    유사도 판단: is_duplicate() 하이브리드 방식
      - 일반: (Jaccard >= 0.7) OR (Cosine >= 0.8)
      - 짧은텍스트 (15자 미만): (Jaccard >= 0.9) OR (Cosine >= 0.95)
    """
    g = group.copy()

    if len(g) == 1:
        return g

    g = g.sort_values('작성일')
    idxs = list(g.index)

    base_idx  = idxs[0]
    base_text = str(g.at[base_idx, TEXT_COL] or '')

    processed = set()

    for idx_j in idxs[1:]:
        if idx_j in processed:
            continue
        comp_text = str(g.at[idx_j, TEXT_COL] or '')

        # 리뷰내용 완전 동일 → 중복
        if base_text == comp_text:
            gid = f"{group_name[:3]}_dup_{counter[0]:06d}"
            counter[0] += 1
            rep = base_idx if len(base_text) >= len(comp_text) else idx_j
            for x in [base_idx, idx_j]:
                g.at[x, '중복_구분']     = '중복'
                g.at[x, '중복_그룹ID']   = gid
                g.at[x, '중복_대표여부'] = (x == rep)
            processed.update([base_idx, idx_j])
            continue

        dup, jac, cos = is_duplicate(base_text, comp_text)

        if dup:
            # 준중복
            gid = f"{group_name[:3]}_quasi_{counter[0]:06d}"
            counter[0] += 1
            rep = base_idx if len(base_text) >= len(comp_text) else idx_j
            sim_val = max(jac, cos)
            for x in [base_idx, idx_j]:
                g.at[x, '중복_구분']     = '준중복'
                g.at[x, '중복_그룹ID']   = gid
                g.at[x, '중복_유사도']   = sim_val
                g.at[x, '중복_대표여부'] = (x == rep)
            processed.update([base_idx, idx_j])
        else:
            # 준중복_복수
            gid = f"{group_name[:3]}_qm_{counter[0]:06d}"
            counter[0] += 1
            for x in [base_idx, idx_j]:
                g.at[x, '중복_구분']     = '준중복_복수'
                g.at[x, '중복_그룹ID']   = gid
                g.at[x, '중복_대표여부'] = True
            processed.update([base_idx, idx_j])

    return g

In [13]:
# 그룹 사이즈 사전 확인
group_sizes = df_main.groupby(['작성자_norm', 'goodsNo', '세션']).size()
print(f"전체 (작성자, 상품, 세션) 그룹 수: {len(group_sizes):,}")
print(f"  단일 리뷰 그룹: {(group_sizes == 1).sum():,}")
print(f"  다중 리뷰 그룹(≥2): {(group_sizes >= 2).sum():,}")
print(f"\n다중 리뷰 그룹 크기 분포:")
print(group_sizes[group_sizes >= 2].value_counts().sort_index().head(10))

전체 (작성자, 상품, 세션) 그룹 수: 526,828
  단일 리뷰 그룹: 425,294
  다중 리뷰 그룹(≥2): 101,534

다중 리뷰 그룹 크기 분포:
2    71897
3    28472
4      734
5      102
6      315
7        5
8        2
9        7
Name: count, dtype: int64


In [14]:
# 단일 그룹 / 다중 그룹 분리 처리 (성능 최적화)
df_main = df_main.merge(
    group_sizes.rename('그룹크기').reset_index(),
    on=['작성자_norm', 'goodsNo', '세션'],
    how='left'
)

mask_single = df_main['그룹크기'] == 1
df_single = df_main[mask_single].copy()
df_multi = df_main[~mask_single].copy()

print(f"단일 그룹 행: {len(df_single):,}")
print(f"다중 그룹 행: {len(df_multi):,}")

# 단일 그룹: 일괄 처리 (빠름)
df_single['action'] = 'keep'
df_single['dup_flag'] = 'unique'
df_single['is_base'] = True
df_single['kept_id'] = df_single['리뷰번호']

# 다중 그룹: process_group 적용 (오래 걸림)
tqdm.pandas(desc="다중 그룹 처리")
df_multi_processed = (
    df_multi.groupby(['작성자_norm', 'goodsNo', '세션'], group_keys=False, sort=False)
            .progress_apply(process_group)
)

# pandas 버전 호환: groupby 키 컬럼이 결과에서 빠진 경우(pandas 3.0+) 인덱스 기반으로 복구
# (process_group 내부에서 원본 정수 인덱스가 보존되므로 가능)
for col in ['작성자_norm', 'goodsNo', '세션']:
    if col not in df_multi_processed.columns:
        df_multi_processed[col] = df_multi.loc[df_multi_processed.index, col]

# 합치기
df_main_processed = pd.concat(
    [df_single, df_multi_processed], ignore_index=True
).sort_values(['작성자_norm', 'goodsNo', '작성일']).reset_index(drop=True)

print(f"\n처리 완료: {len(df_main_processed):,}건")

단일 그룹 행: 425,294
다중 그룹 행: 234,660


다중 그룹 처리:   0%|          | 0/101534 [00:00<?, ?it/s]


처리 완료: 659,954건


---
## 처리 결과 통계

`step2_dedup.py` 실행 결과 (685,025행 처리):

| 분류 | 행 수 | 비율 | 설명 |
|------|------:|-----:|------|
| 완전중복 | 0 | 0.00% | 리뷰번호 포함 완전 동일 (시스템 오류, 0건) |
| 중복 | 723 | 0.11% | IDENTITY 동일 + 리뷰내용 동일 + 24h 이내 |
| 준중복 | 44 | 0.01% | IDENTITY 동일 + 24h 이내 + 유사도 높음 |
| 준중복_복수 | 1,028 | 0.15% | IDENTITY 동일 + 24h 이내 + 유사도 낮음 (모두 유효) |
| 재구매_가능성 | 107,339 | 15.67% | 동일 작성자+상품, 24h 초과 or 구매옵션 다름 |
| **정상** | **575,891** | **84.07%** | 단독 리뷰 |
| **전체** | **685,025** | **100%** | |

- **유효 리뷰 (중복_대표여부=True)**: 684,639행
- **제거 후보 (중복_대표여부=False)**: 386행
- **그룹**: style 104,390행 / non_style 580,635행

In [15]:
print("[action 분포]")
print(df_main_processed['action'].value_counts())
print(f"\n[dup_flag 분포]")
print(df_main_processed['dup_flag'].value_counts(dropna=False))
print(f"\n[is_repurchase 분포]")
print(df_main_processed['is_repurchase'].value_counts())

[action 분포]
action
keep    591696
drop     68258
Name: count, dtype: int64

[dup_flag 분포]
dup_flag
unique                       425294
multi_type_dup               113457
multi_type_unique            107512
multi_option_unique            5917
multi_option_dup               4443
same_option_same_type_dup      1289
multi_both_unique              1263
multi_both_dup                  779
Name: count, dtype: int64

[is_repurchase 분포]
is_repurchase
False    530459
True     129495
Name: count, dtype: int64


In [16]:
# 샘플 확인: drop된 리뷰 vs 보존된 기준 리뷰 페어
print("[multi_option_dup 샘플]")
sample_dup = df_main_processed[
    (df_main_processed['dup_flag'] == 'multi_option_dup') &
    (df_main_processed['action'] == 'drop')
].head(5)
for _, row in sample_dup.iterrows():
    base = df_main_processed[df_main_processed['리뷰번호'] == row['kept_id']].iloc[0]
    print(f"\n  [기준] {row['kept_id']} | 옵션={base['옵션키']} | 한글_글자수={base['한글_글자수']}")
    print(f"     → {base[TEXT_COL][:80]}")
    print(f"  [drop] {row['리뷰번호']} | 옵션={row['옵션키']} | 한글_글자수={row['한글_글자수']}")
    print(f"     → {row[TEXT_COL][:80]}")

[multi_option_dup 샘플]

  [기준] 15343006 | 옵션=('M', '셔츠딥그레이') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합
  [drop] 15342958 | 옵션=('M', '셔츠라이트베이지') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합

  [기준] 15343006 | 옵션=('M', '셔츠딥그레이') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합
  [drop] 15342623 | 옵션=('M', '셔츠블랙') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합

  [기준] 42693190 | 옵션=('2XL', '베이지') | 한글_글자수=43
     → 옷감이 가벼워 여름에 시원하게 입을수 있겠네요뒷주머니가 사진에 안보였는데 진짜로 없어 좀 아쉽네요;;
  [drop] 42693216 | 옵션=('M', '베이지') | 한글_글자수=43
     → 옷감이 가벼워 여름에 시원하게 입을수 있겠네요뒷주머니가 사진에 안보였는데 진짜로 없어 좀 아쉽네요;;

  [기준] 44590746 | 옵션=('XL', '차콜') | 한글_글자수=27
     → 다른색들도 다 맘에들어서 더 샀는데 역시 맘에드네요 감사합니다ㅎㅎ
  [drop] 44590756 | 옵션=('XL', '블랙') | 한글_글자수=27
     → 다른색들도 다 맘에들어서 더 샀는데 역시 맘에드네요 감사합니다ㅎㅎ

  [기준] 44590746

In [17]:
print("[multi_option_unique 샘플 - 둘 다 보존되는 케이스]")
# 동반자(is_base=False)만 순회 → 페어가 한 번씩만 출력됨
companions = (df_main_processed[
    (df_main_processed['dup_flag'] == 'multi_option_unique') & (~df_main_processed['is_base'])
].head(5))
for _, row in companions.iterrows():
    base = df_main_processed[df_main_processed['리뷰번호'] == row['kept_id']].iloc[0]
    print(f"\n  [기준]      {row['kept_id']} | 옵션={base['옵션키']} | 한글_글자수={base['한글_글자수']}")
    print(f"     → {base[TEXT_COL][:100]}")
    print(f"  [같이 보존] {row['리뷰번호']} | 옵션={row['옵션키']} | 한글_글자수={row['한글_글자수']}")
    print(f"     → {row[TEXT_COL][:100]}")

[multi_option_unique 샘플 - 둘 다 보존되는 케이스]

  [기준]      53544349 | 옵션=('M', None) | 한글_글자수=56
     → 먼저 배송이 너무 빨라서 좋았어요! 아주 빠르게 와서 크리스마스 연말 커플룩으로 입을 수 있었어요 :-) 기본 맨투맨으로 입기 딱인 재질입니다!
  [같이 보존] 53544370 | 옵션=('L', None) | 한글_글자수=33
     → 빠른배송이 제일 마음에 들었어요. 목부분 손목부분 쫀쫀해서 그것도 좋았습니당

  [기준]      28940102 | 옵션=('LARGE', None) | 한글_글자수=25
     → 좋아요 두꺼운편인듯 사이즈는 좋아요 환절기에 입어야 할듯
  [같이 보존] 28940077 | 옵션=('MEDIUM', None) | 한글_글자수=21
     → 좋아요 근데 얇지 않습니다 좀 두꺼운편인거 같아요

  [기준]      4753204 | 옵션=('L', '블랙') | 한글_글자수=37
     → 색상 다양하게 구매했는데 전부 마음에 들어요 딱 무난해서 데일리로 잘 입고 다니고 있어요
  [같이 보존] 4753194 | 옵션=('L', '라임') | 한글_글자수=33
     → 색이 너무 마음에 들어요! 쨍한 색감의 티가 갖고 싶었는데 이번기회에 마련했어요

  [기준]      47736657 | 옵션=('L', '네이비') | 한글_글자수=44
     → 일단 옷 재질이 너무 좋습니다 가격대 치고 디자인도 그렇고 두께감도 엄청 두꺼워요 가성비 지리는 옷 입니다
  [같이 보존] 47736574 | 옵션=('M', '네이비') | 한글_글자수=21
     → 엄청 클줄 알고 걱정 했는데 다행이도 잘 맞고 예뻐요

  [기준]      35466465 | 옵션=('S', '(FW)베이지') | 한글_글자수=52
     → 생각했던것보다 색상도 이쁘고 디자인도 맘에들어요 얇은걸 이미 구매한후라 디자인은 이미 알았어요 한겨울은 추울듯합니다
  

---
## month 리뷰 통합 + 저장

- `reviews_step2_dedup.csv`: 최종 보존 리뷰 (Step 3 임베딩 입력)
- `reviews_step2_dropped.csv`: drop된 리뷰 (검증용 보관)

In [18]:
# month 리뷰는 그대로 keep, 플래그만 추가
df_month['action'] = 'keep'
df_month['dup_flag'] = 'month_excluded'
df_month['is_base'] = True
df_month['kept_id'] = df_month['리뷰번호']
df_month['is_repurchase'] = False
df_month['세션'] = 0
df_month['그룹크기'] = 1
df_month['옵션키'] = [
    make_option_key(s, d)
    for s, d in zip(df_month['구매사이즈'], df_month['구매상세'])
]

# 컬럼 정렬을 df_main_processed에 맞춤
df_month = df_month[df_main_processed.columns]

# 통합
df_final = pd.concat([df_main_processed, df_month], ignore_index=True)
print(f"통합 후: {len(df_final):,}건")

통합 후: 685,042건


In [19]:
# 최종 저장: keep만
df_keep = df_final[df_final['action'] == 'keep'].copy()
df_drop = df_final[df_final['action'] == 'drop'].copy()

print(f"최종 보존: {len(df_keep):,}건 ({len(df_keep)/len(df_final)*100:.2f}%)")
print(f"제거된 중복: {len(df_drop):,}건 ({len(df_drop)/len(df_final)*100:.2f}%)")

print(f"\n[보존 리뷰 dup_flag 분포]")
print(df_keep['dup_flag'].value_counts())

print(f"\n[제거된 리뷰 dup_flag 분포]")
print(df_drop['dup_flag'].value_counts())

최종 보존: 616,784건 (90.04%)
제거된 중복: 68,258건 (9.96%)

[보존 리뷰 dup_flag 분포]
dup_flag
unique                       425294
multi_type_unique            107512
multi_type_dup                49008
month_excluded                25088
multi_option_unique            5917
multi_option_dup               2011
multi_both_unique              1263
same_option_same_type_dup       609
multi_both_dup                   82
Name: count, dtype: int64

[제거된 리뷰 dup_flag 분포]
dup_flag
multi_type_dup               64449
multi_option_dup              2432
multi_both_dup                 697
same_option_same_type_dup      680
Name: count, dtype: int64


### 중복_구분 레이블 정의

| 레이블 | 조건 | 중복_대표여부 |
|--------|------|-------------|
| **완전중복** | 리뷰번호 포함 모든 컬럼 동일 | 대표 1개 True, 나머지 False |
| **중복** | IDENTITY + 리뷰내용 동일 + 24h 이내 | 가장 긴 텍스트 True, 나머지 False |
| **준중복** | IDENTITY 동일 + 24h 이내 + (Jac≥0.7 OR Cos≥0.8) | 가장 긴 텍스트 True, 나머지 False |
| **준중복_복수** | IDENTITY 동일 + 24h 이내 + 유사도 낮음 | 모두 True (모두 유효) |
| **재구매_가능성** | 동일 작성자+상품, 24h 초과 or 구매옵션 다름 | 모두 True (모두 유효) |
| **정상** | 단독 리뷰, 위 어느 조건에도 미해당 | True |

### 준중복_복수란?
동일 작성자가 같은 상품에 24시간 이내 여러 리뷰를 작성했지만 **내용이 서로 다른** 케이스.
나중에 텍스트 처리 시 각 리뷰에서 키워드를 추출하여 **리뷰번호 기준으로 합산** 활용 가능.

### 재구매_가능성이란?
동일 작성자가 같은 상품을 **24시간 이후에 다시 구매**하거나 **다른 옵션으로 구매**한 케이스.
진짜 재구매 여부는 추가 분석 필요 (현재는 가능성 플래그로만 분류).

### step2_dedup.py 실행 결과 (2026-04-28 기준)
- 입력: `reviews_step1_cleaned.csv` (685,042행 → 리뷰내용_norm 결측 17행 제거 → 685,025행)
- 출력: `reviews_step2_dedup.csv`
- style 그룹: 104,390행 / non_style 그룹: 580,635행
- 탈퇴자('-') 행: 3,975개 → 각각 고유 ID 부여

In [20]:
# CSV 저장
OUTPUT_KEEP = "reviews_step2_dedup.csv"
OUTPUT_DROP = "reviews_step2_dropped.csv"

df_keep.to_csv(OUTPUT_KEEP, index=False)
df_drop.to_csv(OUTPUT_DROP, index=False)

print(f"저장 완료:")
print(f"  - {OUTPUT_KEEP} ({len(df_keep):,}건)")
print(f"  - {OUTPUT_DROP} ({len(df_drop):,}건)")

저장 완료:
  - reviews_step2_dedup.csv (616,784건)
  - reviews_step2_dropped.csv (68,258건)


---
## (참고) 검증용 분석

전체 처리 결과를 한눈에 점검하기 위한 통계.
실제 분석에는 사용하지 않으므로 필요 시 셀 단위로 참고.

In [21]:
# 같은 작성자 + 같은 상품인데 옵션이 달랐던 케이스 통계
# (base/동반자 모두 같은 플래그를 공유하므로 kept_ 접두사 없음)
multi_opt_pairs = df_keep[df_keep['dup_flag'].isin([
    'multi_option_unique', 'multi_option_dup'
])]
print(f"옵션 다중 구매 관련 보존된 리뷰: {len(multi_opt_pairs):,}건")
print(f"  - base: {multi_opt_pairs['is_base'].sum():,}건")
print(f"  - 동반자(unique 케이스): {(~multi_opt_pairs['is_base']).sum():,}건")

# 재구매 통계
repurchase_keep = df_keep[df_keep['is_repurchase']]
print(f"\n재구매 케이스 보존 리뷰: {len(repurchase_keep):,}건")
print(f"재구매 (작성자, 상품) 쌍: {repurchase_keep.groupby(['작성자_norm', 'goodsNo']).ngroups:,}쌍")

# 브랜드/카테고리별 처리 결과 (만약 컬럼 있으면)
if '브랜드' in df_keep.columns and '카테고리' in df_keep.columns:
    print(f"\n[브랜드 x 카테고리별 보존 리뷰 수]")
    print(df_keep.groupby(['브랜드', '카테고리']).size().unstack(fill_value=0))

옵션 다중 구매 관련 보존된 리뷰: 7,928건
  - base: 4,677건
  - 동반자(unique 케이스): 3,251건

재구매 케이스 보존 리뷰: 118,880건
재구매 (작성자, 상품) 쌍: 50,181쌍


In [22]:
# dup_flag별 샘플 텍스트 확인
import pandas as pd
pd.set_option('display.max_colwidth', None)

SAMPLE_N = 3        # 각 플래그별 표시할 페어/행 수
TEXT_PREVIEW = 120  # 텍스트 미리보기 길이

def print_pair_samples(flag_name, source_df, n=SAMPLE_N):
    """_dup 또는 _unique 페어 출력. 동반자(is_base=False) 행을 골라 base와 매칭."""
    print(f"\n{'='*95}")
    print(f"[dup_flag = {flag_name}]")
    print('='*95)

    companions = source_df[
        (source_df['dup_flag'] == flag_name) & (~source_df['is_base'].fillna(False))
    ]
    if len(companions) == 0:
        print("  (해당 플래그의 동반자 행이 없음)")
        return

    sampled = companions.sample(n=min(n, len(companions)), random_state=42)

    for k, (_, comp) in enumerate(sampled.iterrows(), 1):
        base_match = source_df[source_df['리뷰번호'] == comp['kept_id']]
        if len(base_match) == 0:
            continue
        base = base_match.iloc[0]

        same_opt = base['옵션키'] == comp['옵션키']
        same_typ = base['리뷰타입'] == comp['리뷰타입']
        print(f"\n  ─ 페어 {k} ─ (옵션 {'동일' if same_opt else '다름'} / "
              f"타입 {'동일' if same_typ else '다름'})")
        print(f"    [base]      리뷰번호={base['리뷰번호']} | dup_flag={base['dup_flag']} | "
              f"is_base={base['is_base']} | action={base['action']}")
        print(f"                옵션={base['옵션키']} | 타입={base['리뷰타입']} | "
              f"한글_글자수={base['한글_글자수']}")
        print(f"                → {str(base[TEXT_COL])[:TEXT_PREVIEW]}")
        print(f"    [동반자]    리뷰번호={comp['리뷰번호']} | dup_flag={comp['dup_flag']} | "
              f"is_base={comp['is_base']} | action={comp['action']}")
        print(f"                옵션={comp['옵션키']} | 타입={comp['리뷰타입']} | "
              f"한글_글자수={comp['한글_글자수']}")
        print(f"                → {str(comp[TEXT_COL])[:TEXT_PREVIEW]}")


def print_solo_samples(flag_name, source_df, n=SAMPLE_N):
    """단독 케이스 (unique, month_excluded) 출력."""
    print(f"\n{'='*95}")
    print(f"[dup_flag = {flag_name}]")
    print('='*95)

    pool = source_df[source_df['dup_flag'] == flag_name]
    if len(pool) == 0:
        print("  (해당 플래그 없음)")
        return

    sampled = pool.sample(n=min(n, len(pool)), random_state=42)
    for k, (_, row) in enumerate(sampled.iterrows(), 1):
        print(f"\n  ─ 샘플 {k} ─")
        print(f"    리뷰번호={row['리뷰번호']} | dup_flag={row['dup_flag']} | "
              f"is_base={row['is_base']} | action={row['action']}")
        print(f"    옵션={row['옵션키']} | 타입={row['리뷰타입']} | "
              f"한글_글자수={row['한글_글자수']}")
        print(f"    → {str(row[TEXT_COL])[:TEXT_PREVIEW]}")


# 동반자(drop된 행 포함)도 매칭에 필요하므로 df_final 사용
source = df_final

# _dup 플래그 (페어 출력, base는 keep / 동반자는 drop)
for flag in ['same_option_same_type_dup', 'multi_type_dup',
             'multi_option_dup', 'multi_both_dup']:
    print_pair_samples(flag, source)

# _unique 플래그 (페어 출력, 둘 다 keep)
for flag in ['multi_type_unique', 'multi_option_unique', 'multi_both_unique']:
    print_pair_samples(flag, source)

# 단독 플래그
for flag in ['unique', 'month_excluded']:
    print_solo_samples(flag, source)


[dup_flag = same_option_same_type_dup]

  ─ 페어 1 ─ (옵션 동일 / 타입 동일)
    [base]      리뷰번호=42421340 | dup_flag=multi_type_dup | is_base=True | action=keep
                옵션=('L', '그레이') | 타입=style | 한글_글자수=26
                → 두번째 구매하는데 만족합니다. 두꺼워서 한여름엔 못입습니다
    [동반자]    리뷰번호=42421392 | dup_flag=same_option_same_type_dup | is_base=False | action=drop
                옵션=('L', '그레이') | 타입=style | 한글_글자수=26
                → 두번째 구매하는데 만족합니다. 두꺼워서 한여름엔 못입습니다

  ─ 페어 2 ─ (옵션 동일 / 타입 동일)
    [base]      리뷰번호=9079594 | dup_flag=same_option_same_type_dup | is_base=True | action=keep
                옵션=('L', '블랙') | 타입=goods | 한글_글자수=20
                → 오버핏으로 크게나왔어요 팔은 좀 길지만 좋네요
    [동반자]    리뷰번호=9079596 | dup_flag=same_option_same_type_dup | is_base=False | action=drop
                옵션=('L', '블랙') | 타입=goods | 한글_글자수=20
                → 오버핏으로 크게나왔어요 팔은 좀 길지만 좋네요

  ─ 페어 3 ─ (옵션 동일 / 타입 동일)
    [base]      리뷰번호=33214788 | dup_flag=same_option_same_type_dup | is_base=True | action=keep
     

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. 두 리뷰의 텍스트 추출
text_base = check_df[check_df['리뷰번호'] == 79490920][TEXT_COL].values[0]
text_comp = check_df[check_df['리뷰번호'] == 79490832][TEXT_COL].values[0]

# 2. TF-IDF 벡터화 및 유사도 계산
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform([text_base, text_comp])
similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

print(f"\n{"-"*50}")
print(f"[리뷰 유사도 분석]")
print(f"{"-"*50}")
print(f"  Base (920): {text_base[:]}")
print(f"  Comp (832): {text_comp[:]}")
print(f"\n  코사인 유사도: {similarity:.4f} ({round(similarity * 100, 2)}%)")

# 3. 판단 가이드
if similarity > 0.9:
    print("  결과: 두 리뷰는 거의 동일한 텍스트입니다.")
elif similarity > 0.6:
    print("  결과: 두 리뷰는 상당히 유사한 문장을 공유하고 있습니다.")
else:
    print("  결과: 두 리뷰는 의미적으로 차이가 있거나 구성이 다릅니다.")


--------------------------------------------------
[리뷰 유사도 분석]
--------------------------------------------------
  Base (920): 제품 나쁘지 않네요 크기 적당하고 따뜻한거같아요
  Comp (832): 제품 잘 받았습니다 크기 적당하고 따뜻하네요 굿

  코사인 유사도: 0.3809 (38.09%)
  결과: 두 리뷰는 의미적으로 차이가 있거나 구성이 다릅니다.


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def get_similarity_samples(df, score_min, score_max, n_samples=3):
    """지정된 유사도 구간의 샘플을 추출하여 출력"""
    
    # 1. 비교 대상이 있는(kept_id가 존재하는) 행만 필터링
    target_rows = df[df['kept_id'].notna() & (df['kept_id'] != df['리뷰번호'])].copy()
    
    results = []
    vectorizer = TfidfVectorizer()

    print(f"\n🔎 분석 중: 유사도 구간 {score_min} ~ {score_max} ...")

    for _, row in target_rows.iterrows():
        try:
            # 베이스 리뷰 찾기
            base_review = df[df['리뷰번호'] == row['kept_id']]
            if base_review.empty: continue
            
            text_a = str(base_review.iloc[0][TEXT_COL])
            text_b = str(row[TEXT_COL])
            
            # 유사도 계산
            tfidf = vectorizer.fit_transform([text_a, text_b])
            score = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
            
            # 구간 필터링
            if score_min <= score < score_max:
                results.append({
                    'score': score,
                    'base_id': row['kept_id'],
                    'comp_id': row['리뷰번호'],
                    'base_text': text_a,
                    'comp_text': text_b,
                    'dup_flag': row['dup_flag']
                })
        except:
            continue
            
    # 2. 결과 출력
    sample_df = pd.DataFrame(results)
    if sample_df.empty:
        print(f"   ❌ 해당 구간에 속하는 샘플이 없습니다.")
        return

    sampled = sample_df.sample(n=min(n_samples, len(sample_df)), random_state=42)
    
    for i, res in enumerate(sampled.iterrows(), 1):
        _, data = res
        print(f"\n[{score_min}~{score_max} 구간 샘플 {i}] (Score: {data['score']:.4f})")
        print(f"  - Flag: {data['dup_flag']}")
        print(f"  - Base ({data['base_id']}): {data['base_text'][:100]}...")
        print(f"  - Comp ({data['comp_id']}): {data['comp_text'][:100]}...")
        print("-" * 80)

# --- 실행 ---
# 0.3 ~ 0.35 구간 확인
get_similarity_samples(df_final, 0.3, 0.35, n_samples=3)

# 0.35 ~ 0.4 구간 확인
get_similarity_samples(df_final, 0.35, 0.4, n_samples=3)


🔎 분석 중: 유사도 구간 0.3 ~ 0.35 ...

[0.3~0.35 구간 샘플 1] (Score: 0.3041)
  - Flag: multi_type_unique
  - Base (44927678): 속이 조금 비치지만 그래도 저렴한 맛으로 입어요 굿...
  - Comp (44927597): 핏이 만족스럽지 않으나 저렴한 맛으로 입어요 굿...
--------------------------------------------------------------------------------

[0.3~0.35 구간 샘플 2] (Score: 0.3041)
  - Flag: multi_type_dup
  - Base (8164056): 엄청 얇아요 여름용인듯 그치만 저는 겨울에도 입을거에요...
  - Comp (8164067): 엄청 얇아요 여름용엔듯 하지만 전 겨울에도 입을거에여...
--------------------------------------------------------------------------------

[0.3~0.35 구간 샘플 3] (Score: 0.3041)
  - Flag: multi_type_dup
  - Base (7086522): 174 73인데 m사이즈가니 살짝 오버핏입니다.도톰하고 이뻐요...
  - Comp (7086529): 174 73 m사이즈가니 살짝 오버핏이고 이뻥용~♡...
--------------------------------------------------------------------------------

🔎 분석 중: 유사도 구간 0.35 ~ 0.4 ...

[0.35~0.4 구간 샘플 1] (Score: 0.3685)
  - Flag: multi_type_dup
  - Base (37951089): 많이 바스락거리긴 하지만 색도 예쁘고 크기도 적당하고 두께도 생각보다 도톰해서 좋아요...
  - Comp (37951074): 색이 예쁘고 크기도 적당해서 좋아요. 두께도 생각보다 도톰해요

In [32]:
def get_long_samples_by_similarity(df, score_min, score_max, n_samples=5):
    """지정된 유사도 구간 내에서 문장이 가장 긴 샘플 추출"""
    
    target_rows = df[df['kept_id'].notna() & (df['kept_id'] != df['리뷰번호'])].copy()
    results = []
    vectorizer = TfidfVectorizer()

    print(f"\n🔎 분석 중: 구간 {score_min} ~ {score_max} (긴 문장 우선 추출)...")

    for _, row in target_rows.iterrows():
        try:
            base_review = df[df['리뷰번호'] == row['kept_id']]
            if base_review.empty: continue
            
            text_a = str(base_review.iloc[0][TEXT_COL])
            text_b = str(row[TEXT_COL])
            
            # 유사도 계산
            tfidf = vectorizer.fit_transform([text_a, text_b])
            score = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
            
            if score_min <= score < score_max:
                # 두 문장 길이의 합을 기준으로 저장
                results.append({
                    'score': score,
                    'total_length': len(text_a) + len(text_b),
                    'base_id': row['kept_id'],
                    'comp_id': row['리뷰번호'],
                    'base_text': text_a,
                    'comp_text': text_b,
                    'dup_flag': row['dup_flag']
                })
        except:
            continue
            
    sample_df = pd.DataFrame(results)
    if sample_df.empty:
        print(f"   ❌ 해당 구간에 속하는 샘플이 없습니다.")
        return

    # 💡 정렬 핵심: total_length 기준 내림차순(False) 정렬 후 상위 n개
    longest_samples = sample_df.sort_values(by='total_length', ascending=False).head(n_samples)
    
    for i, (_, data) in enumerate(longest_samples.iterrows(), 1):
        print(f"\n[{score_min}~{score_max} 구간 - 긴 문장 샘플 {i}]")
        print(f"  - Score: {data['score']:.4f} | Total Length: {data['total_length']}")
        print(f"  - Flag: {data['dup_flag']}")
        print(f"  - Base ({data['base_id']}): {data['base_text'][:150]}...") # 길게 보기 위해 150자
        print(f"  - Comp ({data['comp_id']}): {data['comp_text'][:150]}...")
        print("-" * 100)

# --- 실행 ---
# 0.3 ~ 0.35 구간에서 긴 것 5개
get_long_samples_by_similarity(df_final, 0.3, 0.35, n_samples=5)

# 0.35 ~ 0.4 구간에서 긴 것 5개
get_long_samples_by_similarity(df_final, 0.35, 0.4, n_samples=5)


🔎 분석 중: 구간 0.3 ~ 0.35 (긴 문장 우선 추출)...

[0.3~0.35 구간 - 긴 문장 샘플 1]
  - Score: 0.3241 | Total Length: 778
  - Flag: multi_type_dup
  - Base (82084917): 필루미네이트 오버핏 포레스트 체크 셔츠–그린(무신사 단독) 은 은은한 그린 컬러가 매력적인 아이템으로, 튀지 않으면서도 확실한 포인트를 주는 셔츠예요. 오버핏 실루엣이라 어깨선이 자연스럽게 떨어지고 전체적으로 여유 있는 핏이라 체형 부담 없이 편하게 착용할 수 있습니다...
  - Comp (82084942): 필루미네이트 오버핏 포레스트 체크 셔츠–그린(무신사 단독)은 컬러부터 분위기까지 정말 예쁘게 빠진 셔츠예요. 쨍하지 않은 그린 컬러라 부담 없이 입기 좋고, 체크 패턴도 과하지 않아 데일리로 손이 자주 가요. 오버핏이라 어깨선이 자연스럽게 떨어지면서 전체 실루엣이 여유...
----------------------------------------------------------------------------------------------------

[0.3~0.35 구간 - 긴 문장 샘플 2]
  - Score: 0.3307 | Total Length: 507
  - Flag: multi_type_dup
  - Base (53827653): 마감은 사진처럼 가격에비해선 그렇게 썩 좋진 않았는데 일일이 제가 다 대충정리하고 입을만큼 색감때문에 용서가됐다랄까요ㅋㅋ 다른거랑 고민고민하다 너무 색이 밝은거같아서 좀더진할거같은 이녀석으로 픽한건데 증말 옳은선택 한 내자신 매우 칭찬해.. 후기에서처럼 기재된 치수보단...
  - Comp (53813195): 후기에서처럼 수치보단 허리가 좀더 크게나온거 같긴한데 평소 29인치 입는 저로써는 약간 여유있을정도고 수선할정도는 아니라서 정말 다행입니다! 우체국택배라 배송은 확실히 빨라서 좋았네요~! 가격에비해 마감퀄은 좀 아쉬웠는데 맛깔나게 잘빠진 색

In [23]:
# 그룹 크기별 분포 확인
print("[(작성자, 상품, 세션) 그룹 크기 분포]")
print(df_final[df_final['리뷰타입'] != 'month']
      .groupby(['작성자_norm', 'goodsNo', '세션']).size()
      .value_counts().sort_index())

[(작성자, 상품, 세션) 그룹 크기 분포]
1    425294
2     71897
3     28472
4       734
5       102
6       315
7         5
8         2
9         7
Name: count, dtype: int64


In [24]:
# 그룹 크기 ≥ 3인 케이스만 추출해서 그룹 단위로 모든 멤버 표시

LARGE_GROUP_SAMPLE_N = 5  # 출력할 그룹 수

def print_large_group_samples(source_df, min_size=3, n=LARGE_GROUP_SAMPLE_N):
    """그룹 사이즈 ≥ min_size 인 케이스를 그룹 단위로 출력. 모든 멤버를 한 번에 보여줌."""
    
    # month는 별도 처리되므로 제외
    df_check = source_df[source_df['리뷰타입'] != 'month'].copy()
    
    # 그룹 크기 계산
    sizes = df_check.groupby(['작성자_norm', 'goodsNo', '세션']).size()
    large_groups = sizes[sizes >= min_size]
    
    if len(large_groups) == 0:
        print(f"\n그룹 크기 ≥ {min_size}인 케이스 없음")
        return
    
    print(f"\n{'='*95}")
    print(f"[그룹 크기 ≥ {min_size} 케이스: 총 {len(large_groups):,}그룹]")
    print('='*95)
    
    # 그룹 크기 분포
    print("\n그룹 크기별 개수:")
    print(large_groups.value_counts().sort_index().to_string())
    
    # 무작위 n개 그룹 선택
    sample_keys = large_groups.sample(n=min(n, len(large_groups)), 
                                       random_state=42).index.tolist()
    
    for gi, key in enumerate(sample_keys, 1):
        author, goods, sess = key
        members = df_check[
            (df_check['작성자_norm'] == author) &
            (df_check['goodsNo'] == goods) &
            (df_check['세션'] == sess)
        ].sort_values(['is_base', '한글_글자수'], ascending=[False, False])
        
        # 이 그룹 안의 dup_flag 종류 한눈에
        flag_summary = members['dup_flag'].value_counts().to_dict()
        
        print(f"\n{'─'*95}")
        print(f"그룹 {gi}: 작성자={author} | 상품={goods} | 세션={sess} | 멤버 {len(members)}명")
        print(f"  플래그 구성: {flag_summary}")
        print(f"  base 리뷰번호: {members[members['is_base']]['리뷰번호'].iloc[0]}")
        print(f"{'─'*95}")
        
        for mi, (_, m) in enumerate(members.iterrows(), 1):
            role = "★ base" if m['is_base'] else "  동반자"
            print(f"\n  [{role}] 멤버 {mi}")
            print(f"    리뷰번호={m['리뷰번호']} | dup_flag={m['dup_flag']} | "
                  f"is_base={m['is_base']} | action={m['action']}")
            print(f"    옵션={m['옵션키']} | 타입={m['리뷰타입']} | "
                  f"한글_글자수={m['한글_글자수']} | 작성일={m['작성일']}")
            print(f"    → {str(m[TEXT_COL])[:TEXT_PREVIEW]}")


# 그룹 크기별로 따로 보기
print_large_group_samples(source, min_size=3, n=5)   # 3명 이상 그룹 5개
print_large_group_samples(source, min_size=4, n=3)   # 4명 이상 그룹 3개
print_large_group_samples(source, min_size=5, n=2)   # 5명 이상 그룹 2개 (있다면)


[그룹 크기 ≥ 3 케이스: 총 29,637그룹]

그룹 크기별 개수:
3    28472
4      734
5      102
6      315
7        5
8        2
9        7

───────────────────────────────────────────────────────────────────────────────────────────────
그룹 1: 작성자=뮤쉼사 | 상품=1899755 | 세션=1 | 멤버 3명
  플래그 구성: {'multi_type_unique': 3}
  base 리뷰번호: 17989923
───────────────────────────────────────────────────────────────────────────────────────────────

  [★ base] 멤버 1
    리뷰번호=17989923 | dup_flag=multi_type_unique | is_base=True | action=keep
    옵션=('EXTRA LARGE', None) | 타입=photo | 한글_글자수=32 | 작성일=2021-07-03 19:23:29
    → 두깨가 얇지도 두껍지도 않고 적당함 크림색임 약간아이보리 색인데 먼차이냐

  [  동반자] 멤버 2
    리뷰번호=17989900 | dup_flag=multi_type_unique | is_base=False | action=keep
    옵션=('EXTRA LARGE', None) | 타입=style | 한글_글자수=31 | 작성일=2021-07-03 19:22:40
    → 길이가 넣어입고 빼입고 해도 괜찮음 색이 화이트가아니라 크림이라 더 좋음

  [  동반자] 멤버 3
    리뷰번호=17989861 | dup_flag=multi_type_unique | is_base=False | action=keep
    옵션=('EXTRA LARGE', None) | 타입=goods | 한글_글자수=23 | 작성일=202

---
## ⚠️ 알려진 한계: base 행의 dup_flag 의미 모호함

현재 구조에서 그룹 내 여러 동반자가 있고 각 동반자가 서로 다른 dup_flag를 가지는 경우, base 행의 dup_flag는 **그룹 내 첫 번째로 발견된 `_dup` 플래그를 그대로 사용**한다.

### 예시
한 그룹에 멤버 6명이 있고:
- base ↔ 동반자 2: `multi_both_dup` (다른 옵션 + 다른 타입)
- base ↔ 동반자 4: `multi_option_dup` (다른 옵션 + 같은 타입)
- base ↔ 동반자 5: `multi_type_dup` (같은 옵션 + 다른 타입)

이때 base 행의 `dup_flag`는 첫 번째인 `multi_both_dup`으로 표시되지만, 실제로 base는 다른 종류의 관계도 함께 가지고 있다.

### 영향
- **알고리즘 동작에는 문제 없음**: base/동반자 구분은 `is_base` 컬럼이 정확히 처리한다.
- **통계 시 주의**: `df['dup_flag'] == 'multi_both_dup'`로 단순 필터링하면 base와 동반자가 섞이며, base 카운트가 misleading할 수 있다.

### 권장 분석 패턴
페어/플래그 단위 분석 시 **항상 동반자 기준으로 필터링**한다:

```python
# 정확한 multi_both_dup 페어 카운트
df[(df['dup_flag'] == 'multi_both_dup') & (~df['is_base'])]
```

base의 정보가 필요하면 `kept_id`로 매칭하여 가져온다.

### 왜 그대로 두는가
1. 임베딩(Step 3) 입력은 `action == 'keep'`인 모든 행이며, dup_flag 분류 자체에 의존하지 않는다.
2. 분석 단계에서 동반자 기준 필터링 패턴을 지키면 모호함이 발생하지 않는다.

In [25]:
df_keep.head()

,리뷰번호,goodsNo,리뷰타입,작성자,리뷰내용,평점,체험단,구매옵션,키,몸무게,성별,작성일,만족도,사진유무,도움돼요,...,일평균_도움돼요지수,도움여부,리뷰내용_clean,한글_글자수,전체_글자수,is_valid_for_topic,작성자_norm,옵션키,세션,is_repurchase,그룹크기,action,dup_flag,is_base,kept_id
0,34612047,1733275,goods,!!!!!!!!?,요즘 입기 좋은 것 같아요 무난무난하게 잘 입고있습니다,5.0,False,다크그레이 · L,166.0,63.0,여성,2022-11-07 18:13:53,NaN,False,0,...,0.0,0,요즘 입기 좋은 것 같아요 무난무난하게 잘 입고있습니다,23,30,True,!!!!!!!!?,"(L, 다크그레이)",1,False,1,keep,unique,True,34612047
1,51564285,3070563,goods,!!!!!!!!?,잠옷용으로 구매했어요 편하고 그냥 입기에도 조아요,5.0,False,블랙 · 2XL,180.0,90.0,missing,2023-11-23 15:50:39,NaN,False,0,...,0.0,0,잠옷용으로 구매했어요 편하고 그냥 입기에도 조아요,22,27,True,!!!!!!!!?,"(2XL, 블랙)",1,False,1,keep,unique,True,51564285
2,46679669,3251750,goods,!!!!!!!!?,잠옷용으로 휘뚜루마뚜루 입으려고 좀 크게 사긴했는데 진짜 많이 커요 조금은 두꺼운 감이 있어요,5.0,False,블랙 · XL,166.0,50.0,여성,2023-08-01 15:28:26,NaN,False,0,...,0.0,0,잠옷용으로 휘뚜루마뚜루 입으려고 좀 크게 사긴했는데 진짜 많이 커요 조금은 두꺼운 감이 있어요,40,52,True,!!!!!!!!?,"(XL, 블랙)",1,True,1,keep,unique,True,46679669
3,49013088,3251750,goods,!!!!!!!!?,잠옷으로 입으려고 크게 사긴했는데 정말 크고 두꺼워요,5.0,False,블랙 · XL,166.0,50.0,여성,2023-10-01 20:02:46,NaN,False,0,...,0.0,0,잠옷으로 입으려고 크게 사긴했는데 정말 크고 두꺼워요,23,29,True,!!!!!!!!?,"(XL, 블랙)",2,True,1,keep,unique,True,49013088
4,12731504,850153,goods,!!!!!!-,이뻐요. 자주 입겠네요.. 굿 감사랍니다 고맙습니다,5.0,False,블랙 · L,178.0,74.0,남성,2020-11-03 18:57:52,NaN,False,0,...,0.0,0,이뻐요. 자주 입겠네요.. 굿 감사랍니다 고맙습니다,20,28,True,!!!!!!-,"(L, 블랙)",1,False,1,keep,unique,True,12731504
